<a href="https://colab.research.google.com/github/jilanipasha1011/AIAdvocate-Using-LLM-RAG/blob/main/LLM_Fine_Tuning_With_Dolly_DataSet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STEP 1: Google Colab Setup

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# STEP 2: Hugging Face Login

In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# STEP 3: Load Dataset (Dolly 15K)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
dataset = dataset.shuffle(seed=42)

# STEP 4: Format Dataset
Dolly format ko instruction format me convert karo

In [ ]:
def format_dolly(example):
    instruction = example["instruction"]
    context = example["context"] if example["context"] else ""
    response = example["response"]

    text = f"""### Instruction:
{instruction}

### Input:
{context}

### Response:
{response}"""
    return {"text": text}

dataset = dataset.map(format_dolly, remove_columns=dataset.column_names)

# STEP 5: Model & Tokenization

In [ ]:
from transformers import AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Step 6: Load Model (QLoRA 4-bit)

In [ ]:
import torch

# Force disable bf16 globally
torch.backends.cuda.matmul.allow_tf32 = True

# Reload model with explicit FP16 compute
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16  # CRITICAL
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16  # FORCE FP16
)

model.gradient_checkpointing_enable()
model.config.use_cache = False



`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

 # STEP 7: Apply LoRA (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

torch.cuda.empty_cache()

 # STEP 8: Training Setup

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./tinyllama-dolly",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1,
    fp16=True,
    save_strategy="epoch",
    report_to="none"
)

# STEP 9: Trainer Setup

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./tinyllama-dolly",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1,
    fp16=False,   # DISABLE AMP
    bf16=False,   # MUST be False
    packing=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=sft_config,
)

# STEP 10: Start Training

In [ ]:
trainer.train()
torch.cuda.empty_cache()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

# Step 11: Save Model

In [ ]:
trainer.model.save_pretrained("tinyllama-dolly-qlora")
tokenizer.save_pretrained("tinyllama-dolly-qlora")

# STEP 12: Inference Test

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load the base model with the same quantization config used for training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

# Load the LoRA adapters from the local directory onto the base model
model = PeftModel.from_pretrained(base_model, "./tinyllama-dolly-qlora")

# It's good practice to merge the LoRA weights into the base model
# especially if you want to save it as a single model or for slightly faster inference.
# For pipeline usage, passing the PeftModel directly also works.
# model = model.merge_and_unload()

pipe = pipeline(
    "text-generation",
    model=model, # Pass the PeftModel directly to the pipeline
    tokenizer=tokenizer,
    device_map="auto"
)

prompt = """### Instruction:
Explain machine learning in simple terms.

### Input:

### Response:
"""

output = pipe(prompt, max_new_tokens=100, do_sample=True, temperature=0.7)
print(output[0]["generated_text"])

# Now Push On GitHub